In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/2025-sep-dl-gen-ai-project/sample_submission.csv
/kaggle/input/2025-sep-dl-gen-ai-project/train.csv
/kaggle/input/2025-sep-dl-gen-ai-project/test.csv


In [2]:
submission = pd.read_csv('/kaggle/input/2025-sep-dl-gen-ai-project/sample_submission.csv')
train = pd.read_csv('/kaggle/input/2025-sep-dl-gen-ai-project/train.csv')
test = pd.read_csv('/kaggle/input/2025-sep-dl-gen-ai-project/test.csv')

print("submission:\n",submission.head())
print("train:\n",train.head())
print("train:\n",train.describe())
print("train:\n",train.info())

submission:
    id  anger  fear  joy  sadness  surprise
0   0      1     0    0        0         1
1   1      0     1    1        1         0
2   2      1     1    0        0         1
3   3      0     0    0        0         1
4   4      1     1    1        0         0
train:
    id                                               text  anger  fear  joy  \
0   0  the dentist that did the work apparently did a...      1     0    0   
1   1  i'm gonna absolutely ~~suck~~ be terrible duri...      0     1    0   
2   2  bridge: so leave me drowning calling houston, ...      0     1    0   
3   3  after that mess i went to see my now ex-girlfr...      1     1    0   
4   4  as he stumbled i ran off, afraid it might some...      0     1    0   

   sadness  surprise                    emotions  
0        1         0         ['anger' 'sadness']  
1        1         0          ['fear' 'sadness']  
2        1         0          ['fear' 'sadness']  
3        1         0  ['anger' 'fear' 'sadness']

In [3]:
train['emotions'] = train['emotions'].to_list()
train['emotions_count'] = train['emotions'].apply(len)
print(train[train['emotions_count']==2].shape)
print(train[(train['joy']==1) & (train['sadness']==1)].shape)
surprise_count = train[(train['surprise']==1)].shape[0]
total_count = train.shape[0]
print('surprise percentage: ', round((surprise_count/total_count)*100,0))


(676, 9)
(96, 9)
surprise percentage:  29.0


In [4]:
print('anger_count: ',train[(train['anger']==1)].shape[0])
print('fear_count: ',train[(train['fear']==1)].shape[0])
print('joy_count: ',train[(train['joy']==1)].shape[0])
print('sadness_count: ',train[(train['sadness']==1)].shape[0])
print('surprise_count: ',train[(train['surprise']==1)].shape[0])

anger_count:  808
fear_count:  3860
joy_count:  1660
sadness_count:  2171
surprise_count:  1999


In [5]:
import re

def text_word_length(text):
    words = re.findall(r'\b\w+\b', text)
    lengths = [len(word) for word in words]
    return sum(lengths)

train['text_word_length'] = train['text'].apply(text_word_length)

print(train.head())
print(train.describe())
print(train.info())


   id                                               text  anger  fear  joy  \
0   0  the dentist that did the work apparently did a...      1     0    0   
1   1  i'm gonna absolutely ~~suck~~ be terrible duri...      0     1    0   
2   2  bridge: so leave me drowning calling houston, ...      0     1    0   
3   3  after that mess i went to see my now ex-girlfr...      1     1    0   
4   4  as he stumbled i ran off, afraid it might some...      0     1    0   

   sadness  surprise                    emotions  emotions_count  \
0        1         0         ['anger' 'sadness']              19   
1        1         0          ['fear' 'sadness']              18   
2        1         0          ['fear' 'sadness']              18   
3        1         0  ['anger' 'fear' 'sadness']              26   
4        0         0                    ['fear']               8   

   text_word_length  
0               121  
1                61  
2               115  
3                83  
4           

In [6]:
train[['anger','fear','joy','sadness','surprise']].corr().round(2)

,anger,fear,joy,sadness,surprise
anger,1.00,0.08,-0.19,0.09,0.02
fear,0.08,1.00,-0.47,0.29,0.16
joy,-0.19,-0.47,1.00,-0.32,-0.10
sadness,0.09,0.29,-0.32,1.00,-0.12
surprise,0.02,0.16,-0.10,-0.12,1.00


In [7]:
import string

def normalize_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    return ''.join(char for char in text if char not in string.punctuation)

train['normalized_text'] = train['text'].apply(normalize_text)
train.head()

train['normalized_text_length'] = train['normalized_text'].apply(len)
train['text_length'] = train['text'].apply(len)

print(sum(train['normalized_text_length']))
print(sum(train['text_length']))
print('Percentage Change: ', (sum(train['text_length'])-sum(train['normalized_text_length']))/(sum(train['text_length']))*100)

524580
542274
Percentage Change:  3.2629261222186567


In [8]:
from nltk.corpus import stopwords
import nltk
from collections import Counter

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

all_text = ' '.join(train['text'].dropna()).lower()

words = re.findall(r'\b\w+\b', all_text)
unique_words_list = list(set(words))
    
# Count unique words
unique_word_count = len(set(unique_words_list))
print("Unique word count:", unique_word_count)

stopword_count = sum(1 for word in unique_words_list if word in stop_words)
print("Stop-word count:", stopword_count)

print("Stop-word percentage:", (stopword_count/unique_word_count)*100)

filtered_words = [word for word in words if word not in stop_words]

word_counts = Counter(filtered_words)

most_common_words = word_counts.most_common(5)
fifth_word = most_common_words[4][0] if len(most_common_words) >= 5 else None

print("5th most frequent word:", fifth_word)
print(word_counts)

Unique word count: 8035
Stop-word count: 146
Stop-word percentage: 1.8170504044803983
5th most frequent word: heart
Counter({'head': 545, 'eyes': 441, 'like': 407, 'back': 373, 'heart': 343, 'one': 330, 'face': 295, 'get': 291, 'time': 278, 'still': 272, 'got': 243, 'never': 221, 'hands': 219, 'really': 206, 'hand': 200, 'felt': 196, 'know': 195, 'went': 189, 'day': 189, 'could': 189, 'little': 180, 'around': 170, 'started': 166, 'going': 165, 'see': 159, 'feet': 155, 'right': 153, 'legs': 146, 'mouth': 142, 'feel': 142, 'way': 138, 'away': 135, 'night': 132, 'first': 129, 'go': 127, 'even': 122, 'room': 119, 'put': 116, 'stomach': 116, 'well': 114, 'saw': 111, 'good': 109, 'think': 107, 'something': 107, 'said': 105, 'look': 104, 'last': 104, 'home': 103, 'thing': 103, 'much': 102, 'brain': 98, 'chest': 96, 'left': 95, 'thought': 94, 'took': 92, 'foot': 91, 'door': 91, 'feeling': 91, 'two': 90, 'open': 90, 'getting': 90, 'nothing': 89, 'bit': 89, 'also': 88, 'arms': 87, 'though': 86, 

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

all_text = ' '.join(train['text'].dropna()).lower()

text_data = re.findall(r'\b\w+\b', all_text)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit and transform the text data
tfidf_matrix = vectorizer.fit_transform(text_data)
print(tfidf_matrix.shape)

# Convert to a dense matrix and view feature names
feature_names = vectorizer.get_feature_names_out()
dense_matrix = tfidf_matrix.toarray()

# Display results
print("Feature Names:\n", feature_names)
print("\nTF-IDF Matrix:\n", dense_matrix)


(107990, 7722)
Feature Names:
 ['00' '000' '02' ... 'zombie' 'zone' 'zooming']

TF-IDF Matrix:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# Step 1: Vectorize the text
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
X = vectorizer.fit_transform(train['text'])

test_data = vectorizer.transform(test['text'])  # DO NOT use fit_transform here

# Step 2: Extract emotion columns as multi-label targets
y = train[['anger', 'fear', 'joy', 'sadness', 'surprise']]

# Step 3: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 4: Train Decision Tree for each emotion label
# We'll train one classifier per emotion and count total nodes
total_nodes = 0
for emotion in y.columns:
    clf = DecisionTreeClassifier(random_state=42, max_depth=6)
    clf.fit(X_train, y_train[emotion])
    total_nodes += clf.tree_.node_count
    print(f"{emotion}: {clf.tree_.node_count} nodes")

# Step 5: Total nodes across all classifiers
print("Total nodes across all emotion classifiers:", total_nodes)

f1_scores = []
for emotion in y.columns:
    clf = DecisionTreeClassifier(random_state=42, max_depth=6)
    clf.fit(X_train, y_train[emotion])
    y_pred = clf.predict(X_train)
    score = f1_score(y_train[emotion], y_pred, average='macro')
    f1_scores.append(score)
    print(f"{emotion} F1 score: {score:.4f}")

# Step 5: Macro F1 across all emotions
macro_f1 = sum(f1_scores) / len(f1_scores)
print(f"\nMacro F1 score on training split: {macro_f1:.4f}")

# Step 4: Train classifiers and compute Macro F1 on test split
f1_scores = []
for emotion in y.columns:
    clf = DecisionTreeClassifier(random_state=42, max_depth=6)
    clf.fit(X_train, y_train[emotion])
    y_pred = clf.predict(X_test)
    score = f1_score(y_test[emotion], y_pred, average='macro')
    f1_scores.append(score)
    print(f"{emotion} F1 score (test): {score:.4f}")

# Step 5: Macro F1 across all emotions
macro_f1 = sum(f1_scores) / len(f1_scores)
print(f"\nMacro F1 score on test split: {macro_f1:.4f}")


anger: 25 nodes
fear: 27 nodes
joy: 29 nodes
sadness: 35 nodes
surprise: 45 nodes
Total nodes across all emotion classifiers: 161
anger F1 score: 0.5521
fear F1 score: 0.4096
joy F1 score: 0.5203
sadness F1 score: 0.5076
surprise F1 score: 0.4870

Macro F1 score on training split: 0.4953
anger F1 score (test): 0.5324
fear F1 score (test): 0.3735
joy F1 score (test): 0.4937
sadness F1 score (test): 0.4856
surprise F1 score (test): 0.4843

Macro F1 score on test split: 0.4739


In [11]:
# Step 4: Train classifiers and compute Macro F1 on test split
for emotion in ['anger', 'fear', 'joy', 'sadness', 'surprise']:
    clf = DecisionTreeClassifier(random_state=42, max_depth=6)
    clf.fit(X_train, y_train[emotion])
    y_pred = clf.predict(test_data)
    submission[emotion]=y_pred

submission.head

<bound method NDFrame.head of         id  anger  fear  joy  sadness  surprise
0        0      0     0    0        0         0
1        1      0     1    0        0         0
2        2      0     1    0        0         0
3        3      0     1    0        0         0
4        4      0     1    0        0         0
...    ...    ...   ...  ...      ...       ...
1702  1702      0     1    0        0         0
1703  1703      0     1    0        0         0
1704  1704      0     1    0        0         0
1705  1705      0     1    0        0         0
1706  1706      0     1    0        1         0

[1707 rows x 6 columns]>